# Extração — DATASUS (Raw -> Bronze)

Este notebook lê os 4 CSV do DATASUS já carregados na camada **Raw** do MinIO (bucket `raw`, prefixo `dados_brutos/csv/DATASUS/`) — exports do TabNet com procedimentos hospitalares (SIH/SUS) e de produção ambulatorial (SIA/SUS) — estrutura em formato tabular (long/tidy) e grava o resultado em Parquet na camada **Bronze**, na subpasta `dados_DATASUS`, localmente em `dados_processados/bronze/dados_DATASUS/` e no MinIO (bucket `bronze`, prefixo `dados_DATASUS/`).

**Por que esses CSV também precisam de estruturação antes de virar Parquet:** assim como os exports SIDRA do IBGE ([extracao_bronze_ibge.ipynb](extracao_bronze_ibge.ipynb)), o TabNet exporta uma tabela pensada para leitura humana, não para consumo tabular direto: 3 linhas de título antes do cabeçalho, cada ano como uma coluna própria (formato largo), o código do procedimento e sua descrição concatenados numa única célula, um valor `-` para "sem dado", e um rodapé de texto livre (fonte, notas) depois da última linha real. `pd.read_csv()` direto não separaria nada disso — por isso a extração aqui interpreta a estrutura do arquivo antes de gerar uma tabela tidy: uma linha por (procedimento, ano).

**Sobre a codificação:** diferente dos CSV do IBGE (UTF-8 com BOM), estes vêm em **Latin-1** (`ISO-8859-1`) — o padrão dos exports legados do TabNet. Ler com o encoding errado não geraria um erro imediato, geraria acentos corrompidos silenciosamente em todo o arquivo; por isso o encoding é declarado explicitamente, não deixado no padrão do sistema.

## Imports

In [1]:
import csv
import io
import os
import re
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from minio import Minio

## Configuração

Origem: bucket `raw`, prefixo `dados_brutos/csv/DATASUS/`. Destino: bucket `bronze` + pasta local `dados_processados/bronze/dados_DATASUS/` — mesma convenção de dupla gravação já usada nas Bronze do ISAPS e do IBGE.

In [2]:
load_dotenv(Path.cwd().parent / ".env")

MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT", "localhost:9000")
MINIO_ACCESS_KEY = os.getenv("MINIO_ACCESS_KEY")
MINIO_SECRET_KEY = os.getenv("MINIO_SECRET_KEY")

BUCKET_RAW = os.getenv("BUCKET_RAW", "raw")
RAW_CSV_PREFIX = "dados_brutos/csv/DATASUS/"

BUCKET_BRONZE = os.getenv("BUCKET_BRONZE", "bronze")
BRONZE_PREFIX = "dados_DATASUS/"

BRONZE_DIR = Path.cwd().parent / "dados_processados" / "bronze" / "dados_DATASUS"
BRONZE_DIR.mkdir(parents=True, exist_ok=True)

client = Minio(MINIO_ENDPOINT, access_key=MINIO_ACCESS_KEY, secret_key=MINIO_SECRET_KEY, secure=False)
if not client.bucket_exists(BUCKET_BRONZE):
    client.make_bucket(BUCKET_BRONZE)
    print(f"Bucket '{BUCKET_BRONZE}' criado.")

print(f"MinIO: {MINIO_ENDPOINT} | bucket raw: {BUCKET_RAW} | bucket bronze: {BUCKET_BRONZE}")
print(f"Saida parquet local: {BRONZE_DIR}")

MinIO: localhost:9000 | bucket raw: raw | bucket bronze: bronze
Saida parquet local: C:\Projeto_AI\dados_processados\bronze\dados_DATASUS


## Funções de extração

Os 4 exports do TabNet seguem sempre a mesma anatomia, nas linhas (0-indexado):

| Linha | Conteúdo |
|---|---|
| 0 | Título do relatório (ex.: `"Procedimentos hospitalares do SUS - por local de internação - Brasil"`) |
| 1 | Descrição da variável (ex.: `"AIH aprovadas por Procedimento e Ano atendimento"`) |
| 2 | Período coberto (ex.: `"Período:2014-2024"`) |
| 3 | Cabeçalho: `Procedimento`, um ano por coluna, e por último `Total` |
| 4+ | Linhas de dado — cada uma é um procedimento, até a linha `Procedimento == "Total"` (a soma de todos os procedimentos por ano) |

Depois da linha `Total` vem o rodapé de texto livre (fonte, notas) que não representa nenhuma linha de tabela.

**Duas decisões estruturais, nos mesmos moldes já usados na Bronze do ISAPS e do IBGE:**

- **A coluna `Total`** (soma de todos os anos para aquele procedimento) é **excluída** do formato longo — é um agregado por linha, não um ano real, e incluí-la faria `ano` conter um valor não numérico. É o mesmo motivo pelo qual a extração do ISAPS excluía a coluna `WORLDWIDE`: um agregado na mesma dimensão dos dados reais, mas que não é um membro válido dela.
- **A linha `Total`** (soma de todos os procedimentos para aquele ano) é **mantida** — continua sendo um dado real por ano, só que agregado, exatamente como as linhas `Total Procedures` foram mantidas na Bronze do ISAPS. Ela fica identificável por não ter `procedimento_codigo` (ver função `parse_procedimento` abaixo).

**Sobre separar código e nome do procedimento:** cada célula da coluna `Procedimento` vem como `"<código SUS de 10 dígitos> <descrição>"` (ex.: `"0201010038 BIOPSIA CIRURGICA DE TIREOIDE"`) numa única string. Separar isso em `procedimento_codigo` e `procedimento_nome` é a mesma categoria de decomposição estrutural já aplicada na Bronze do IBGE (`periodo_original` -> `ano`/`trimestre`): mecânica, sem juízo de negócio, e o texto original (`procedimento_original`) é preservado para conferência.

In [3]:
def parse_number(v):
    if v is None:
        return None
    s = str(v).strip()
    if s in ("", "-"):
        return None
    s = s.replace(".", "").replace(",", ".")
    try:
        return float(s)
    except ValueError:
        return None


def parse_procedimento(label):
    """Separa 'procedimento_original' em codigo (10 digitos) e nome. A linha agregada
    'Total' nao tem codigo, entao fica com procedimento_codigo=None."""
    m = re.match(r"^(\d{10}) (.+)$", label)
    if m:
        return m.group(1), m.group(2)
    return None, label


def ler_linhas_csv(texto):
    return list(csv.reader(io.StringIO(texto), delimiter=";", quotechar='"'))


def parse_tabela_datasus(texto, arquivo_origem, categoria, metrica):
    linhas = ler_linhas_csv(texto)

    titulo = linhas[0][0].strip()
    descricao_variavel = linhas[1][0].strip()
    periodo_relatorio = linhas[2][0].strip()
    header = linhas[3]
    anos_colunas = header[1:-1]  # exclui "Procedimento" (primeira) e "Total" (ultima)

    registros = []
    for row in linhas[4:]:
        if not row or len(row) != len(header):
            break
        label = row[0].strip()
        codigo, nome = parse_procedimento(label)
        for ano_str, valor_str in zip(anos_colunas, row[1:-1]):
            registros.append({
                "arquivo_origem": arquivo_origem,
                "categoria": categoria,
                "metrica": metrica,
                "titulo": titulo,
                "descricao_variavel": descricao_variavel,
                "periodo_relatorio": periodo_relatorio,
                "procedimento_original": label,
                "procedimento_codigo": codigo,
                "procedimento_nome": nome,
                "ano": int(ano_str),
                "quantidade": parse_number(valor_str),
            })
        if label == "Total":
            break

    return pd.DataFrame(registros)

## Configuração por arquivo e extração

Cada CSV representa uma métrica diferente sobre procedimentos do SUS — duas do sistema hospitalar (SIH/SUS) e duas do sistema ambulatorial (SIA/SUS) — então, como já feito para o IBGE, cada um vira o seu próprio Parquet em vez de um único arquivo consolidado.

In [4]:
ARQUIVOS_DATASUS = {
    "AIH-aprovadas_Procedimentos-hospitalares.csv": {"categoria": "hospitalar", "metrica": "aih_aprovadas"},
    "Valor-total_Procedimentos-hospitalares.csv": {"categoria": "hospitalar", "metrica": "valor_total"},
    "Qtd.aprovada_Producao-Ambulatorial.csv": {"categoria": "ambulatorial", "metrica": "qtd_aprovada"},
    "Valor-aprovado_Producao-Ambulatorial.csv": {"categoria": "ambulatorial", "metrica": "valor_aprovado"},
}

tabelas_bronze = {}
for fname, cfg in ARQUIVOS_DATASUS.items():
    object_name = f"{RAW_CSV_PREFIX}{fname}"
    texto = client.get_object(BUCKET_RAW, object_name).read().decode("latin-1")
    df = parse_tabela_datasus(texto, fname, cfg["categoria"], cfg["metrica"])
    tabelas_bronze[fname] = df
    print(f"{fname}: {len(df)} linhas, {df['ano'].nunique()} anos, {df['procedimento_codigo'].nunique()} procedimentos")

AIH-aprovadas_Procedimentos-hospitalares.csv: 32094 linhas, 18 anos, 1782 procedimentos
Valor-total_Procedimentos-hospitalares.csv: 32094 linhas, 18 anos, 1782 procedimentos


Qtd.aprovada_Producao-Ambulatorial.csv: 33876 linhas, 12 anos, 2822 procedimentos


Valor-aprovado_Producao-Ambulatorial.csv: 26604 linhas, 12 anos, 2216 procedimentos


## Prévia das tabelas em formato Bronze (sem tratamento/limpeza)

In [5]:
for fname, df in tabelas_bronze.items():
    print(f"\n=== {fname} ({len(df)} linhas) ===")
    display(df.head(5))
    display(df[df['procedimento_codigo'].isnull()].head(3))  # confere a linha 'Total'


=== AIH-aprovadas_Procedimentos-hospitalares.csv (32094 linhas) ===


,arquivo_origem,categoria,metrica,titulo,descricao_variavel,periodo_relatorio,procedimento_original,procedimento_codigo,procedimento_nome,ano,quantidade
0,AIH-aprovadas_Procedimentos-hospitalares.csv,hospitalar,aih_aprovadas,Procedimentos hospitalares do SUS - por local ...,AIH aprovadas por Procedimento e Ano atendimento,Período:2014-2024,0201010038 BIOPSIA CIRURGICA DE TIREOIDE,0201010038,BIOPSIA CIRURGICA DE TIREOIDE,2007,NaN
1,AIH-aprovadas_Procedimentos-hospitalares.csv,hospitalar,aih_aprovadas,Procedimentos hospitalares do SUS - por local ...,AIH aprovadas por Procedimento e Ano atendimento,Período:2014-2024,0201010038 BIOPSIA CIRURGICA DE TIREOIDE,0201010038,BIOPSIA CIRURGICA DE TIREOIDE,2008,NaN
2,AIH-aprovadas_Procedimentos-hospitalares.csv,hospitalar,aih_aprovadas,Procedimentos hospitalares do SUS - por local ...,AIH aprovadas por Procedimento e Ano atendimento,Período:2014-2024,0201010038 BIOPSIA CIRURGICA DE TIREOIDE,0201010038,BIOPSIA CIRURGICA DE TIREOIDE,2009,NaN
3,AIH-aprovadas_Procedimentos-hospitalares.csv,hospitalar,aih_aprovadas,Procedimentos hospitalares do SUS - por local ...,AIH aprovadas por Procedimento e Ano atendimento,Período:2014-2024,0201010038 BIOPSIA CIRURGICA DE TIREOIDE,0201010038,BIOPSIA CIRURGICA DE TIREOIDE,2010,NaN
4,AIH-aprovadas_Procedimentos-hospitalares.csv,hospitalar,aih_aprovadas,Procedimentos hospitalares do SUS - por local ...,AIH aprovadas por Procedimento e Ano atendimento,Período:2014-2024,0201010038 BIOPSIA CIRURGICA DE TIREOIDE,0201010038,BIOPSIA CIRURGICA DE TIREOIDE,2011,NaN


,arquivo_origem,categoria,metrica,titulo,descricao_variavel,periodo_relatorio,procedimento_original,procedimento_codigo,procedimento_nome,ano,quantidade
32076,AIH-aprovadas_Procedimentos-hospitalares.csv,hospitalar,aih_aprovadas,Procedimentos hospitalares do SUS - por local ...,AIH aprovadas por Procedimento e Ano atendimento,Período:2014-2024,Total,NaN,Total,2007,14.0
32077,AIH-aprovadas_Procedimentos-hospitalares.csv,hospitalar,aih_aprovadas,Procedimentos hospitalares do SUS - por local ...,AIH aprovadas por Procedimento e Ano atendimento,Período:2014-2024,Total,NaN,Total,2008,311684.0
32078,AIH-aprovadas_Procedimentos-hospitalares.csv,hospitalar,aih_aprovadas,Procedimentos hospitalares do SUS - por local ...,AIH aprovadas por Procedimento e Ano atendimento,Período:2014-2024,Total,NaN,Total,2009,24577.0



=== Valor-total_Procedimentos-hospitalares.csv (32094 linhas) ===


,arquivo_origem,categoria,metrica,titulo,descricao_variavel,periodo_relatorio,procedimento_original,procedimento_codigo,procedimento_nome,ano,quantidade
0,Valor-total_Procedimentos-hospitalares.csv,hospitalar,valor_total,Procedimentos hospitalares do SUS - por local ...,Valor total por Procedimento e Ano atendimento,Período:2014-2024,0201010038 BIOPSIA CIRURGICA DE TIREOIDE,0201010038,BIOPSIA CIRURGICA DE TIREOIDE,2007,NaN
1,Valor-total_Procedimentos-hospitalares.csv,hospitalar,valor_total,Procedimentos hospitalares do SUS - por local ...,Valor total por Procedimento e Ano atendimento,Período:2014-2024,0201010038 BIOPSIA CIRURGICA DE TIREOIDE,0201010038,BIOPSIA CIRURGICA DE TIREOIDE,2008,NaN
2,Valor-total_Procedimentos-hospitalares.csv,hospitalar,valor_total,Procedimentos hospitalares do SUS - por local ...,Valor total por Procedimento e Ano atendimento,Período:2014-2024,0201010038 BIOPSIA CIRURGICA DE TIREOIDE,0201010038,BIOPSIA CIRURGICA DE TIREOIDE,2009,NaN
3,Valor-total_Procedimentos-hospitalares.csv,hospitalar,valor_total,Procedimentos hospitalares do SUS - por local ...,Valor total por Procedimento e Ano atendimento,Período:2014-2024,0201010038 BIOPSIA CIRURGICA DE TIREOIDE,0201010038,BIOPSIA CIRURGICA DE TIREOIDE,2010,NaN
4,Valor-total_Procedimentos-hospitalares.csv,hospitalar,valor_total,Procedimentos hospitalares do SUS - por local ...,Valor total por Procedimento e Ano atendimento,Período:2014-2024,0201010038 BIOPSIA CIRURGICA DE TIREOIDE,0201010038,BIOPSIA CIRURGICA DE TIREOIDE,2011,NaN


...

,arquivo_origem,categoria,metrica,titulo,descricao_variavel,periodo_relatorio,procedimento_original,procedimento_codigo,procedimento_nome,ano,quantidade
0,Qtd.aprovada_Producao-Ambulatorial.csv,ambulatorial,qtd_aprovada,Produção Ambulatorial do SUS - Brasil - por lo...,Qtd.aprovada por Procedimento e Ano atendimento,Período:2014-2024,0101010010 ATIVIDADE EDUCATIVA / ORIENTACAO EM...,0101010010,ATIVIDADE EDUCATIVA / ORIENTACAO EM GRUPO NA A...,2013,952017.0
1,Qtd.aprovada_Producao-Ambulatorial.csv,ambulatorial,qtd_aprovada,Produção Ambulatorial do SUS - Brasil - por lo...,Qtd.aprovada por Procedimento e Ano atendimento,Período:2014-2024,0101010010 ATIVIDADE EDUCATIVA / ORIENTACAO EM...,0101010010,ATIVIDADE EDUCATIVA / ORIENTACAO EM GRUPO NA A...,2014,53650772.0
2,Qtd.aprovada_Producao-Ambulatorial.csv,ambulatorial,qtd_aprovada,Produção Ambulatorial do SUS - Brasil - por lo...,Qtd.aprovada por Procedimento e Ano atendimento,Período:2014-2024,0101010010 ATIVIDADE EDUCATIVA / ORIENTACAO EM...,0101010010,ATIVIDADE EDUCATIVA / ORIENTACAO EM GRUPO NA A...,2015,47248142.0
3,Qtd.aprovada_Producao-Ambulatorial.csv,ambulatorial,qtd_aprovada,Produção Ambulatorial do SUS - Brasil - por lo...,Qtd.aprovada por Procedimento e Ano atendimento,Período:2014-2024,0101010010 ATIVIDADE EDUCATIVA / ORIENTACAO EM...,0101010010,ATIVIDADE EDUCATIVA / ORIENTACAO EM GRUPO NA A...,2016,38411159.0
4,Qtd.aprovada_Producao-Ambulatorial.csv,ambulatorial,qtd_aprovada,Produção Ambulatorial do SUS - Brasil - por lo...,Qtd.aprovada por Procedimento e Ano atendimento,Período:2014-2024,0101010010 ATIVIDADE EDUCATIVA / ORIENTACAO EM...,0101010010,ATIVIDADE EDUCATIVA / ORIENTACAO EM GRUPO NA A...,2017,26088770.0


,arquivo_origem,categoria,metrica,titulo,descricao_variavel,periodo_relatorio,procedimento_original,procedimento_codigo,procedimento_nome,ano,quantidade
33864,Qtd.aprovada_Producao-Ambulatorial.csv,ambulatorial,qtd_aprovada,Produção Ambulatorial do SUS - Brasil - por lo...,Qtd.aprovada por Procedimento e Ano atendimento,Período:2014-2024,Total,NaN,Total,2013,1.767852e+07
33865,Qtd.aprovada_Producao-Ambulatorial.csv,ambulatorial,qtd_aprovada,Produção Ambulatorial do SUS - Brasil - por lo...,Qtd.aprovada por Procedimento e Ano atendimento,Período:2014-2024,Total,NaN,Total,2014,4.091225e+09
33866,Qtd.aprovada_Producao-Ambulatorial.csv,ambulatorial,qtd_aprovada,Produção Ambulatorial do SUS - Brasil - por lo...,Qtd.aprovada por Procedimento e Ano atendimento,Período:2014-2024,Total,NaN,Total,2015,4.113909e+09



=== Valor-aprovado_Producao-Ambulatorial.csv (26604 linhas) ===


,arquivo_origem,categoria,metrica,titulo,descricao_variavel,periodo_relatorio,procedimento_original,procedimento_codigo,procedimento_nome,ano,quantidade
0,Valor-aprovado_Producao-Ambulatorial.csv,ambulatorial,valor_aprovado,Produção Ambulatorial do SUS - Brasil - por lo...,Valor aprovado por Procedimento e Ano atendimento,Período:2014-2024,0101010028 ATIVIDADE EDUCATIVA / ORIENTACAO EM...,0101010028,ATIVIDADE EDUCATIVA / ORIENTACAO EM GRUPO NA A...,2013,27868.2
1,Valor-aprovado_Producao-Ambulatorial.csv,ambulatorial,valor_aprovado,Produção Ambulatorial do SUS - Brasil - por lo...,Valor aprovado por Procedimento e Ano atendimento,Período:2014-2024,0101010028 ATIVIDADE EDUCATIVA / ORIENTACAO EM...,0101010028,ATIVIDADE EDUCATIVA / ORIENTACAO EM GRUPO NA A...,2014,9065822.6
2,Valor-aprovado_Producao-Ambulatorial.csv,ambulatorial,valor_aprovado,Produção Ambulatorial do SUS - Brasil - por lo...,Valor aprovado por Procedimento e Ano atendimento,Período:2014-2024,0101010028 ATIVIDADE EDUCATIVA / ORIENTACAO EM...,0101010028,ATIVIDADE EDUCATIVA / ORIENTACAO EM GRUPO NA A...,2015,11609139.6
3,Valor-aprovado_Producao-Ambulatorial.csv,ambulatorial,valor_aprovado,Produção Ambulatorial do SUS - Brasil - por lo...,Valor aprovado por Procedimento e Ano atendimento,Período:2014-2024,0101010028 ATIVIDADE EDUCATIVA / ORIENTACAO EM...,0101010028,ATIVIDADE EDUCATIVA / ORIENTACAO EM GRUPO NA A...,2016,9667847.8
4,Valor-aprovado_Producao-Ambulatorial.csv,ambulatorial,valor_aprovado,Produção Ambulatorial do SUS - Brasil - por lo...,Valor aprovado por Procedimento e Ano atendimento,Período:2014-2024,0101010028 ATIVIDADE EDUCATIVA / ORIENTACAO EM...,0101010028,ATIVIDADE EDUCATIVA / ORIENTACAO EM GRUPO NA A...,2017,9939117.2


,arquivo_origem,categoria,metrica,titulo,descricao_variavel,periodo_relatorio,procedimento_original,procedimento_codigo,procedimento_nome,ano,quantidade
26592,Valor-aprovado_Producao-Ambulatorial.csv,ambulatorial,valor_aprovado,Produção Ambulatorial do SUS - Brasil - por lo...,Valor aprovado por Procedimento e Ano atendimento,Período:2014-2024,Total,NaN,Total,2013,8.592228e+07
26593,Valor-aprovado_Producao-Ambulatorial.csv,ambulatorial,valor_aprovado,Produção Ambulatorial do SUS - Brasil - por lo...,Valor aprovado por Procedimento e Ano atendimento,Período:2014-2024,Total,NaN,Total,2014,1.741569e+10
26594,Valor-aprovado_Producao-Ambulatorial.csv,ambulatorial,valor_aprovado,Produção Ambulatorial do SUS - Brasil - por lo...,Valor aprovado por Procedimento e Ano atendimento,Período:2014-2024,Total,NaN,Total,2015,1.779391e+10


## Conversão para Parquet (camada Bronze, subpasta `dados_DATASUS`)

Grava cada tabela em `dados_processados/bronze/dados_DATASUS/{nome}.parquet` e envia ao MinIO no bucket `bronze`, sob o prefixo `dados_DATASUS/`.

In [6]:
arquivos_gerados = {}
for fname, df in tabelas_bronze.items():
    slug = re.sub(r"[^a-z0-9]+", "_", Path(fname).stem.lower()).strip("_")
    out_name = f"{slug}.parquet"
    out_path = BRONZE_DIR / out_name

    df.to_parquet(out_path, engine="pyarrow", index=False)
    arquivos_gerados[fname] = out_path

    object_name = f"{BRONZE_PREFIX}{out_name}"
    client.fput_object(BUCKET_BRONZE, object_name, str(out_path))
    print(f"Gravado: {out_path} ({len(df)} linhas) -> s3://{BUCKET_BRONZE}/{object_name}")

Gravado: C:\Projeto_AI\dados_processados\bronze\dados_DATASUS\aih_aprovadas_procedimentos_hospitalares.parquet (32094 linhas) -> s3://bronze/dados_DATASUS/aih_aprovadas_procedimentos_hospitalares.parquet


Gravado: C:\Projeto_AI\dados_processados\bronze\dados_DATASUS\valor_total_procedimentos_hospitalares.parquet (32094 linhas) -> s3://bronze/dados_DATASUS/valor_total_procedimentos_hospitalares.parquet


Gravado: C:\Projeto_AI\dados_processados\bronze\dados_DATASUS\qtd_aprovada_producao_ambulatorial.parquet (33876 linhas) -> s3://bronze/dados_DATASUS/qtd_aprovada_producao_ambulatorial.parquet


Gravado: C:\Projeto_AI\dados_processados\bronze\dados_DATASUS\valor_aprovado_producao_ambulatorial.parquet (26604 linhas) -> s3://bronze/dados_DATASUS/valor_aprovado_producao_ambulatorial.parquet


## Conferência final

Relê os Parquet recém-gravados do disco para confirmar que o conteúdo persistido bate com o esperado — a mesma checagem de "ida e volta" (write, depois read) usada nas conferências finais dos demais notebooks de extração.

In [7]:
for fname, out_path in arquivos_gerados.items():
    df = pd.read_parquet(out_path)
    print(f"{out_path.name}: {df.shape}")
    display(df.sample(min(5, len(df)), random_state=42))

aih_aprovadas_procedimentos_hospitalares.parquet: (32094, 11)


,arquivo_origem,categoria,metrica,titulo,descricao_variavel,periodo_relatorio,procedimento_original,procedimento_codigo,procedimento_nome,ano,quantidade
26772,AIH-aprovadas_Procedimentos-hospitalares.csv,hospitalar,aih_aprovadas,Procedimentos hospitalares do SUS - por local ...,AIH aprovadas por Procedimento e Ano atendimento,Período:2014-2024,0410010146 RETIRADA DE PROTESE MAMARIA BILATER...,0410010146,RETIRADA DE PROTESE MAMARIA BILATERAL EM CASOS...,2013,1.0
26164,AIH-aprovadas_Procedimentos-hospitalares.csv,hospitalar,aih_aprovadas,Procedimentos hospitalares do SUS - por local ...,AIH aprovadas por Procedimento e Ano atendimento,Período:2014-2024,0409070033 COLPOCLEISE (CIRURGIA DE LE FORT),0409070033,COLPOCLEISE (CIRURGIA DE LE FORT),2017,346.0
24832,AIH-aprovadas_Procedimentos-hospitalares.csv,hospitalar,aih_aprovadas,Procedimentos hospitalares do SUS - por local ...,AIH aprovadas por Procedimento e Ano atendimento,Período:2014-2024,0409020109 RESSECCAO E FECHAMENTO DE FISTULA U...,0409020109,RESSECCAO E FECHAMENTO DE FISTULA URETRAL,2017,290.0
17779,AIH-aprovadas_Procedimentos-hospitalares.csv,hospitalar,aih_aprovadas,Procedimentos hospitalares do SUS - por local ...,AIH aprovadas por Procedimento e Ano atendimento,Período:2014-2024,0408020105 FASCIOTOMIA DE MEMBROS SUPERIORES,0408020105,FASCIOTOMIA DE MEMBROS SUPERIORES,2020,612.0
8729,AIH-aprovadas_Procedimentos-hospitalares.csv,hospitalar,aih_aprovadas,Procedimentos hospitalares do SUS - por local ...,AIH aprovadas por Procedimento e Ano atendimento,Período:2014-2024,0404020640 TRATAMENTO CIRURGICO DE ANQUILOSE D...,0404020640,TRATAMENTO CIRURGICO DE ANQUILOSE DA ARTICULAC...,2024,144.0


valor_total_procedimentos_hospitalares.parquet: (32094, 11)


,arquivo_origem,categoria,metrica,titulo,descricao_variavel,periodo_relatorio,procedimento_original,procedimento_codigo,procedimento_nome,ano,quantidade
26772,Valor-total_Procedimentos-hospitalares.csv,hospitalar,valor_total,Procedimentos hospitalares do SUS - por local ...,Valor total por Procedimento e Ano atendimento,Período:2014-2024,0410010146 RETIRADA DE PROTESE MAMARIA BILATER...,0410010146,RETIRADA DE PROTESE MAMARIA BILATERAL EM CASOS...,2013,580.00
26164,Valor-total_Procedimentos-hospitalares.csv,hospitalar,valor_total,Procedimentos hospitalares do SUS - por local ...,Valor total por Procedimento e Ano atendimento,Período:2014-2024,0409070033 COLPOCLEISE (CIRURGIA DE LE FORT),0409070033,COLPOCLEISE (CIRURGIA DE LE FORT),2017,140763.69
24832,Valor-total_Procedimentos-hospitalares.csv,hospitalar,valor_total,Procedimentos hospitalares do SUS - por local ...,Valor total por Procedimento e Ano atendimento,Período:2014-2024,0409020109 RESSECCAO E FECHAMENTO DE FISTULA U...,0409020109,RESSECCAO E FECHAMENTO DE FISTULA URETRAL,2017,129981.73
17779,Valor-total_Procedimentos-hospitalares.csv,hospitalar,valor_total,Procedimentos hospitalares do SUS - por local ...,Valor total por Procedimento e Ano atendimento,Período:2014-2024,0408020105 FASCIOTOMIA DE MEMBROS SUPERIORES,0408020105,FASCIOTOMIA DE MEMBROS SUPERIORES,2020,393340.94
8729,Valor-total_Procedimentos-hospitalares.csv,hospitalar,valor_total,Procedimentos hospitalares do SUS - por local ...,Valor total por Procedimento e Ano atendimento,Período:2014-2024,0404020640 TRATAMENTO CIRURGICO DE ANQUILOSE D...,0404020640,TRATAMENTO CIRURGICO DE ANQUILOSE DA ARTICULAC...,2024,110183.18


qtd_aprovada_producao_ambulatorial.parquet: (33876, 11)


,arquivo_origem,categoria,metrica,titulo,descricao_variavel,periodo_relatorio,procedimento_original,procedimento_codigo,procedimento_nome,ano,quantidade
12620,Qtd.aprovada_Producao-Ambulatorial.csv,ambulatorial,qtd_aprovada,Produção Ambulatorial do SUS - Brasil - por lo...,Qtd.aprovada por Procedimento e Ano atendimento,Período:2014-2024,0211090050 DETERMINACAO DE PRESSAO INTRA-ABDOM...,0211090050,DETERMINACAO DE PRESSAO INTRA-ABDOMINAL,2021,2694.0
25167,Qtd.aprovada_Producao-Ambulatorial.csv,ambulatorial,qtd_aprovada,Produção Ambulatorial do SUS - Brasil - por lo...,Qtd.aprovada por Procedimento e Ano atendimento,Período:2014-2024,0501010017 COLETA DE SANGUE EM HEMOCENTRO P/ E...,0501010017,COLETA DE SANGUE EM HEMOCENTRO P/ EXAMES DE HI...,2016,281511.0
21614,Qtd.aprovada_Producao-Ambulatorial.csv,ambulatorial,qtd_aprovada,Produção Ambulatorial do SUS - Brasil - por lo...,Qtd.aprovada por Procedimento e Ano atendimento,Período:2014-2024,0401020088 EXERESE DE CISTO SACRO-COCCIGEO,0401020088,EXERESE DE CISTO SACRO-COCCIGEO,2015,44.0
17877,Qtd.aprovada_Producao-Ambulatorial.csv,ambulatorial,qtd_aprovada,Produção Ambulatorial do SUS - Brasil - por lo...,Qtd.aprovada por Procedimento e Ano atendimento,Período:2014-2024,0304010448 RADIOTERAPIA DE PENIS,0304010448,RADIOTERAPIA DE PENIS,2022,163.0
775,Qtd.aprovada_Producao-Ambulatorial.csv,ambulatorial,qtd_aprovada,Produção Ambulatorial do SUS - Brasil - por lo...,Qtd.aprovada por Procedimento e Ano atendimento,Período:2014-2024,0102010323 LICENCIAMENTO SANITARIO DE INDUSTRI...,0102010323,LICENCIAMENTO SANITARIO DE INDUSTRIAS DE MEDIC...,2020,542.0


valor_aprovado_producao_ambulatorial.parquet: (26604, 11)


,arquivo_origem,categoria,metrica,titulo,descricao_variavel,periodo_relatorio,procedimento_original,procedimento_codigo,procedimento_nome,ano,quantidade
8477,Valor-aprovado_Producao-Ambulatorial.csv,ambulatorial,valor_aprovado,Produção Ambulatorial do SUS - Brasil - por lo...,Valor aprovado por Procedimento e Ano atendimento,Período:2014-2024,0204060117 RADIOGRAFIA DE COXA,0204060117,RADIOGRAFIA DE COXA,2018,6153464.97
13597,Valor-aprovado_Producao-Ambulatorial.csv,ambulatorial,valor_aprovado,Produção Ambulatorial do SUS - Brasil - por lo...,Valor aprovado por Procedimento e Ano atendimento,Período:2014-2024,0301130035 ACOMPANHAMENTO NO PROCESSO TRANSEXU...,0301130035,ACOMPANHAMENTO NO PROCESSO TRANSEXUALIZADO EXC...,2014,16263.94
196,Valor-aprovado_Producao-Ambulatorial.csv,ambulatorial,valor_aprovado,Produção Ambulatorial do SUS - Brasil - por lo...,Valor aprovado por Procedimento e Ano atendimento,Período:2014-2024,0102010234 RECEBIMENTO DE DENUNCIAS/RECLAMACOES,0102010234,RECEBIMENTO DE DENUNCIAS/RECLAMACOES,2017,14501.99
15443,Valor-aprovado_Producao-Ambulatorial.csv,ambulatorial,valor_aprovado,Produção Ambulatorial do SUS - Brasil - por lo...,Valor aprovado por Procedimento e Ano atendimento,Período:2014-2024,0304020222 QUIMIOTERAPIA DO CARCINOMA PULMONAR...,0304020222,QUIMIOTERAPIA DO CARCINOMA PULMONAR INDIFERENC...,2024,6176500.00
7923,Valor-aprovado_Producao-Ambulatorial.csv,ambulatorial,valor_aprovado,Produção Ambulatorial do SUS - Brasil - por lo...,Valor aprovado por Procedimento e Ano atendimento,Período:2014-2024,0204030129 RADIOGRAFIA DE TORAX (APICO-LORDORT...,0204030129,RADIOGRAFIA DE TORAX (APICO-LORDORTICA),2016,609839.30
